# 02 — Train NER Model
Fine-tunes a BERT-based token classifier to detect `EXP_DATE`, `START_DATE`, and `DURATION` spans.
**Requires:** `01_build_dataset.ipynb` to have been run first (creates `../data/deadline_sentences/`).

## Cell 1 — Imports

In [ ]:
import os, re, time, platform, random
import torch
import torch.nn as nn
import numpy as np
from collections import Counter
from datasets import load_from_disk
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification,
    EarlyStoppingCallback,
)
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report
import mlflow, mlflow.pytorch

print('PyTorch     :', torch.__version__)
print('CUDA avail  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU         :', torch.cuda.get_device_name(0))
print('Imports OK')

## Cell 2 — MLflow & Environment Config

In [ ]:
os.environ['AWS_ACCESS_KEY_ID']      = 'datanauts-key'
os.environ['AWS_SECRET_ACCESS_KEY']  = 'datanauts-secret'
os.environ['MLFLOW_S3_ENDPOINT_URL'] = 'http://129.114.27.190:9000'
os.environ['GIT_PYTHON_REFRESH']     = 'quiet'

MLFLOW_URI = 'http://129.114.27.190:8000'
EXPERIMENT = 'deadline-detection-ner'
DATA_PATH  = '../data/deadline_sentences'
OUTPUT_DIR = '/tmp/deadline-ner'

mlflow.set_tracking_uri(MLFLOW_URI)
print('MLflow URI  :', MLFLOW_URI)
print('Data path   :', DATA_PATH)

## Cell 3 — Label Schema & Model Configs
Change `MODEL_VARIANT` here to switch between training runs.

In [ ]:
LABEL_LIST = ['O','B-EXP_DATE','I-EXP_DATE','B-START_DATE','I-START_DATE','B-DURATION','I-DURATION']
LABEL2ID   = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL   = {i: l for i, l in enumerate(LABEL_LIST)}

CONFIGS = {
    'baseline':       {'base_model': None,                  'epochs': 0, 'learning_rate': 0,    'batch_size': 16, 'max_seq_length': 256, 'fp16': False, 'entity_weight': 1.0},
    'bert_ner_v1':    {'base_model': 'dslim/bert-base-NER', 'epochs': 3, 'learning_rate': 2e-5, 'batch_size': 16, 'max_seq_length': 256, 'fp16': True,  'entity_weight': 1.0},
    'bert_ner_v2':    {'base_model': 'dslim/bert-base-NER', 'epochs': 3, 'learning_rate': 5e-5, 'batch_size': 16, 'max_seq_length': 256, 'fp16': True,  'entity_weight': 1.0},
    'bert_ner_v3':    {'base_model': 'dslim/bert-base-NER', 'epochs': 5, 'learning_rate': 2e-5, 'batch_size': 16, 'max_seq_length': 256, 'fp16': True,  'entity_weight': 1.0},
    'bert_base_cased':{'base_model': 'bert-base-cased',     'epochs': 3, 'learning_rate': 2e-5, 'batch_size': 16, 'max_seq_length': 256, 'fp16': True,  'entity_weight': 1.0},
    'bert_ner_v4':    {'base_model': 'dslim/bert-base-NER', 'epochs': 5, 'learning_rate': 2e-5, 'batch_size': 16, 'max_seq_length': 256, 'fp16': True,  'entity_weight': 50.0},
}

# ── CHANGE THIS to switch variants ──────────────────────────────────────────
MODEL_VARIANT = 'bert_ner_v4'
# ────────────────────────────────────────────────────────────────────────────

cfg = CONFIGS[MODEL_VARIANT]
print(f'Selected variant : {MODEL_VARIANT}')
print(f'Config           : {cfg}')
print(f'Entity weight    : {cfg["entity_weight"]}x  (1.0 = standard, 50.0 = penalise missed entities 50x)')

## Cell 4 — Seed & Load Data

In [ ]:
def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seeds(42)

dd       = load_from_disk(DATA_PATH)
train_ds = dd['train'].select_columns(['tokens', 'ner_tags'])
val_ds   = dd['val'].select_columns(['tokens', 'ner_tags'])
test_ds  = dd['test'].select_columns(['tokens', 'ner_tags'])

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)} sentences')

all_tags = [t for ex in train_ds for t in ex['ner_tags']]
tag_counts = Counter(all_tags)
print('\nNER tag counts (train):')
for i, label in enumerate(LABEL_LIST):
    print(f'  {label:<16}: {tag_counts.get(i, 0):>7}')

## Cell 5 — Load Tokenizer & Tokenize Dataset

In [ ]:
if MODEL_VARIANT == 'baseline':
    print('Baseline mode — skipping tokenization')
    tokenizer = None
    tok_train = tok_val = tok_test = None
else:
    tokenizer = AutoTokenizer.from_pretrained(cfg['base_model'])
    print(f'Loaded tokenizer: {cfg["base_model"]}')

    def tokenize_and_align(examples):
        tokenized = tokenizer(
            examples['tokens'], truncation=True,
            max_length=cfg['max_seq_length'], is_split_into_words=True,
        )
        aligned = []
        for i, labels in enumerate(examples['ner_tags']):
            word_ids = tokenized.word_ids(batch_index=i)
            prev, ids = None, []
            for wid in word_ids:
                if wid is None:     ids.append(-100)
                elif wid != prev:   ids.append(labels[wid])
                else:               ids.append(-100)
                prev = wid
            aligned.append(ids)
        tokenized['labels'] = aligned
        return tokenized

    tok_train = train_ds.map(tokenize_and_align, batched=True, remove_columns=train_ds.column_names)
    tok_val   = val_ds.map(tokenize_and_align,   batched=True, remove_columns=val_ds.column_names)
    tok_test  = test_ds.map(tokenize_and_align,  batched=True, remove_columns=test_ds.column_names)
    print(f'Tokenized — train: {len(tok_train)} | val: {len(tok_val)} | test: {len(tok_test)}')

    ex = tok_train[0]
    tokens_decoded = tokenizer.convert_ids_to_tokens(ex['input_ids'])
    labels_readable = [ID2LABEL[l] if l != -100 else '[sub]' for l in ex['labels']]
    print('\nSample tokenized row (first 20 tokens):')
    print(list(zip(tokens_decoded[:20], labels_readable[:20])))

## Cell 6 — Regex Baseline (runs for all variants as reference)

In [ ]:
MONTHS = {
    'january','february','march','april','may','june',
    'july','august','september','october','november','december',
    'jan','feb','mar','apr','jun','jul','aug','sep','oct','nov','dec',
}

def run_baseline(ds):
    true_seqs, pred_seqs = [], []
    for sample in ds:
        tokens   = sample['tokens']
        tags     = sample['ner_tags']
        true_seq = [ID2LABEL[t] for t in tags]
        pred_seq, i = [], 0
        while i < len(tokens):
            tok_clean = tokens[i].rstrip('.,').lower()
            if tok_clean in MONTHS:
                pred_seq.append('B-EXP_DATE')
                i += 1
                while i < len(tokens) and re.match(r'^\d{1,4}[,.]?$', tokens[i]):
                    pred_seq.append('I-EXP_DATE')
                    i += 1
            else:
                pred_seq.append('O')
                i += 1
        true_seqs.append(true_seq)
        pred_seqs.append(pred_seq)
    return {
        'f1':        f1_score(true_seqs, pred_seqs),
        'precision': precision_score(true_seqs, pred_seqs),
        'recall':    recall_score(true_seqs, pred_seqs),
    }

print('Running regex baseline on test set...')
baseline_metrics = run_baseline(test_ds)
print(f"Baseline F1        : {baseline_metrics['f1']:.4f}")
print(f"Baseline Precision : {baseline_metrics['precision']:.4f}")
print(f"Baseline Recall    : {baseline_metrics['recall']:.4f}")

## Cell 7 — WeightedNERTrainer
Used when `entity_weight > 1.0` (i.e. `bert_ner_v4`). Upweights B-/I- entity token loss to force the model to find rare entities instead of defaulting to O.
> Standard `Trainer` is automatically used for v1/v2/v3 where `entity_weight = 1.0`.

In [ ]:
class WeightedNERTrainer(Trainer):
    """
    Custom Trainer that upweights entity token loss to counter O-token dominance.
    O tokens keep weight 1.0. All B-/I- entity classes get entity_weight.
    """
    def __init__(self, entity_weight, num_labels, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.entity_weight = entity_weight
        self.num_labels    = num_labels

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits
        weights        = torch.ones(self.num_labels, device=logits.device)
        weights[1:]    = self.entity_weight
        loss_fn = nn.CrossEntropyLoss(weight=weights, ignore_index=-100)
        loss    = loss_fn(logits.view(-1, self.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

print('WeightedNERTrainer defined')
print(f'For {MODEL_VARIANT}: entity_weight = {cfg["entity_weight"]}')
if cfg['entity_weight'] > 1.0:
    print(f'  → Missing an entity token costs {cfg["entity_weight"]}x more than misclassifying an O token')
else:
    print(f'  → Standard uniform loss (no upweighting)')

## Cell 8 — Train Model
> Skip this cell if `MODEL_VARIANT = 'baseline'`

In [ ]:
def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)
    true_preds  = [[ID2LABEL[x] for x, l in zip(pred, lab) if l != -100] for pred, lab in zip(preds, labels)]
    true_labels = [[ID2LABEL[l] for l in lab if l != -100] for lab in labels]
    return {
        'f1':        f1_score(true_labels, true_preds),
        'precision': precision_score(true_labels, true_preds),
        'recall':    recall_score(true_labels, true_preds),
    }

if MODEL_VARIANT == 'baseline':
    print('Baseline variant — no model training needed.')
    model = None; train_result = None; train_time = 0
else:
    model = AutoModelForTokenClassification.from_pretrained(
        cfg['base_model'], num_labels=len(LABEL_LIST),
        id2label=ID2LABEL, label2id=LABEL2ID, ignore_mismatched_sizes=True,
    )
    t_args = TrainingArguments(
        output_dir=f'{OUTPUT_DIR}-{MODEL_VARIANT}',
        num_train_epochs=cfg['epochs'],
        per_device_train_batch_size=cfg['batch_size'],
        per_device_eval_batch_size=cfg['batch_size'],
        learning_rate=cfg['learning_rate'],
        weight_decay=0.01,
        warmup_steps=50,
        fp16=cfg['fp16'] and torch.cuda.is_available(),
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        logging_steps=20,
        report_to='none',
    )

    if cfg['entity_weight'] > 1.0:
        print(f'Using WeightedNERTrainer (entity_weight={cfg["entity_weight"]})')
        trainer = WeightedNERTrainer(
            entity_weight=cfg['entity_weight'],
            num_labels=len(LABEL_LIST),
            model=model, args=t_args,
            train_dataset=tok_train, eval_dataset=tok_val,
            processing_class=tokenizer,
            data_collator=DataCollatorForTokenClassification(tokenizer),
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        )
    else:
        print(f'Using standard Trainer (uniform loss)')
        trainer = Trainer(
            model=model, args=t_args,
            train_dataset=tok_train, eval_dataset=tok_val,
            processing_class=tokenizer,
            data_collator=DataCollatorForTokenClassification(tokenizer),
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        )

    print(f'Training {MODEL_VARIANT}  |  lr={cfg["learning_rate"]}  |  epochs={cfg["epochs"]}  |  fp16={cfg["fp16"] and torch.cuda.is_available()}')
    t0 = time.time()
    train_result = trainer.train()
    train_time = time.time() - t0
    print(f'\nTraining complete in {train_time:.1f}s')
    print(f'Final train loss: {train_result.training_loss:.4f}')

## Cell 9 — Evaluate on Test Set

In [ ]:
if MODEL_VARIANT == 'baseline':
    test_result = {'eval_f1': baseline_metrics['f1'],
                   'eval_precision': baseline_metrics['precision'],
                   'eval_recall': baseline_metrics['recall'],
                   'eval_loss': 0}
    report = ''
    print(f'Baseline — F1: {baseline_metrics["f1"]:.4f}')
else:
    print('Evaluating on test set...')
    preds_out   = trainer.predict(tok_test)
    raw_preds   = np.argmax(preds_out.predictions, axis=2)
    raw_labs    = preds_out.label_ids
    true_preds  = [[ID2LABEL[x] for x, l in zip(p, lb) if l != -100] for p, lb in zip(raw_preds, raw_labs)]
    true_labels = [[ID2LABEL[l] for l in lb if l != -100] for lb in raw_labs]
    report = classification_report(true_labels, true_preds)
    test_result = {
        'eval_f1':        f1_score(true_labels, true_preds),
        'eval_precision': precision_score(true_labels, true_preds),
        'eval_recall':    recall_score(true_labels, true_preds),
        'eval_loss':      preds_out.metrics.get('test_loss', 0),
    }
    print(report)
    print(f'Test F1        : {test_result["eval_f1"]:.4f}')
    print(f'Test Precision : {test_result["eval_precision"]:.4f}')
    print(f'Test Recall    : {test_result["eval_recall"]:.4f}')
    print(f'Baseline F1    : {baseline_metrics["f1"]:.4f}  (delta: {test_result["eval_f1"]-baseline_metrics["f1"]:+.4f})')

## Cell 10 — Log to MLflow

In [ ]:
mlflow.set_experiment(EXPERIMENT)
run_name = MODEL_VARIANT if MODEL_VARIANT != 'baseline' else 'ner_baseline_regex'

with mlflow.start_run(run_name=run_name) as run:
    mlflow.log_params({
        'model_name':     run_name,
        'base_model':     cfg.get('base_model', 'regex') or 'regex',
        'epochs':         cfg['epochs'],
        'batch_size':     cfg['batch_size'],
        'learning_rate':  cfg['learning_rate'],
        'max_seq_length': cfg['max_seq_length'],
        'fp16':           cfg['fp16'],
        'entity_weight':  cfg['entity_weight'],
        'num_labels':     len(LABEL_LIST),
        'label_schema':   'EXP_DATE|START_DATE|DURATION',
        'train_size':     len(train_ds),
        'val_size':       len(val_ds),
        'test_size':      len(test_ds),
        'gpu':            torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'pytorch':        torch.__version__,
        'platform':       platform.platform(),
    })
    mlflow.log_metrics({
        'baseline_f1':          baseline_metrics['f1'],
        'total_train_time_sec': train_time if MODEL_VARIANT != 'baseline' else 0,
        'train_loss':           train_result.training_loss if train_result else 0,
        'test_f1':              test_result['eval_f1'],
        'test_precision':       test_result['eval_precision'],
        'test_recall':          test_result['eval_recall'],
        'test_loss':            test_result['eval_loss'],
    })
    if report:
        mlflow.log_text(report, 'ner_classification_report.txt')

    run_url = f'{MLFLOW_URI}/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}'
    print(f'Logged run : {run_name}')
    print(f'MLflow URL : {run_url}')